<a href="https://colab.research.google.com/github/polreig/StartUp_DecoAI/blob/main/DecoAI_V2.0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Instalación

In [ ]:
!pip install -q diffusers transformers accelerate opencv-python Pillow gradio pydantic google-generativeai huggingface_hub

Súper motor Versión 2.0

In [ ]:
import gradio as gr
import torch
import cv2
import json
import numpy as np
import urllib.parse
from PIL import Image, ImageDraw
from pydantic import BaseModel, Field
from typing import List, Tuple

from google import genai
from google.genai import types
from google.colab import userdata
from diffusers import StableDiffusionControlNetInpaintPipeline, ControlNetModel, DDIMScheduler
from transformers import AutoImageProcessor, Mask2FormerForUniversalSegmentation

print("🚀 INICIANDO DECO.AI (V3.0 INTEGRAL: LAYOUT PRO)...")

# --- 1. CREDENCIALES ---
try:
    GOOGLE_API_KEY = userdata.get('clave_API_gemini')
    gemini_client = genai.Client(api_key=GOOGLE_API_KEY)
except Exception as e:
    print("⚠️ ADVERTENCIA: No se encontró 'clave_API_gemini'.")

# --- 2. CARGA CENTRALIZADA DE MODELOS (SD 1.5 + ControlNet MLSD) ---
print("📏 Cargando IA de Segmentación...")
processor = AutoImageProcessor.from_pretrained("facebook/mask2former-swin-tiny-coco-panoptic")
segmentation_model = Mask2FormerForUniversalSegmentation.from_pretrained(
    "facebook/mask2former-swin-tiny-coco-panoptic", torch_dtype=torch.float16
).to("cuda")

print("📐 Cargando IA de Perspectiva Estructural (MLSD)...")
controlnet = ControlNetModel.from_pretrained("lllyasviel/control_v11p_sd15_mlsd", torch_dtype=torch.float16)

print("🎨 Cargando Motor Gráfico (SD1.5 Inpainting Pro)...")
pipe = StableDiffusionControlNetInpaintPipeline.from_pretrained(
    "runwayml/stable-diffusion-inpainting",
    controlnet=controlnet,
    torch_dtype=torch.float16
).to("cuda")
pipe.scheduler = DDIMScheduler.from_config(pipe.scheduler.config)


# --- 3. ESQUEMAS DE DATOS ---
class ProductoRecomendado(BaseModel):
    nombre: str = Field(description="Nombre descriptivo del producto real")
    categoria: str = Field(description="'mueble', 'iluminacion', 'suelo', 'pared', 'electrodomestico', 'decoracion'")
    tienda_sugerida: str = Field(description="Leroy Merlin, IKEA, Amazon, Zara Home, Bauhaus, Kave Home, Maisons du Monde")
    precio_estimado_eur: int
    detalles_medidas: str

class AnalisisDecoracionPro(BaseModel):
    modo_generacion: str = Field(description="'completo', 'parcial', 'mover'")
    prompt_generacion: str = Field(description="Prompt en inglés detallado")
    presupuesto_total_estimado: int
    lista_compra: List[ProductoRecomendado]


# --- 4. FUNCIONES CORE ---
def redimensionar(img, target_size=512):
    """SD1.5 trabaja mejor en 512x512 nativo en Colab T4"""
    ancho, alto = img.size
    ratio = alto / ancho
    nuevo_ancho, nuevo_alto = (target_size, int(target_size * ratio)) if ancho > alto else (int(max_size / ratio), max_size)
    nuevo_ancho, nuevo_alto = (nuevo_ancho // 8) * 8, (nuevo_alto // 8) * 8
    return img.resize((nuevo_ancho, nuevo_alto), Image.Resampling.LANCZOS)

def extraer_mlsd(img):
    img_gray = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2GRAY)
    lsd = cv2.createLineSegmentDetector(0)
    lines, _, _, _ = lsd.detect(img_gray)
    drawn_img = np.zeros_like(img_gray)
    if lines is not None: lsd.drawSegments(drawn_img, lines)
    return Image.fromarray(drawn_img).convert("RGB")

def traducir_prompt(texto_es):
    if not texto_es.strip(): return ""
    prompt = f"Traduce este texto de diseño de interiores al inglés para usarlo en Stable Diffusion. Responde SOLO con la traducción directa, sin comillas: '{texto_es}'"
    response = gemini_client.models.generate_content(model='gemini-2.5-flash', contents=prompt)
    return response.text.strip()

def generar_html_tienda(datos):
    lista_ordenada = sorted(datos['lista_compra'], key=lambda x: x['precio_estimado_eur'])
    html = f"<div style='background:#f8f9fa; padding:25px; border-radius:12px; border: 1px solid #e9ecef; font-family: sans-serif;'><div style='display: flex; justify-content: space-between; border-bottom: 2px solid #dee2e6; padding-bottom: 10px; margin-bottom: 15px;'><h3 style='margin: 0; color: #212529;'>🧾 Presupuesto Estimado</h3><h3 style='margin: 0; color: #28a745;'>~{datos['presupuesto_total_estimado']}€</h3></div><ul>"
    
    for item in lista_ordenada:
        query = urllib.parse.quote(item['nombre'])
        html += f"<li style='margin-bottom:12px; padding:15px; background:white; border-radius:8px; display: flex; justify-content: space-between;'><div><b>{item['nombre'].title()}</b><br><span style='font-size: 12px;'>📏 {item['detalles_medidas']} | 🏬 Sugerencia: {item['tienda_sugerida']}</span></div><div style='text-align: right;'><span style='font-weight: bold;'>~{item['precio_estimado_eur']}€</span><br><a href='https://www.google.com/search?q=comprar+{query}' target='_blank' style='background:#212529; color:white; padding:4px 8px; border-radius:4px; text-decoration:none; font-size:12px;'>Ver Oferta</a></div></li>"
    html += "</ul></div>"
    return html


# --- 5. LÓGICA DE LAS PESTAÑAS (V3.0 PILARES) ---

# Pilar 1: Completo
def motor_pilar_1_completo(img_entrada, peticion_es):
    if img_entrada is None: return None, "Sube una imagen primero."
    img_orig = redimensionar(Image.fromarray(img_entrada).convert("RGB"))
    
    prompt_gemini = f"Cliente pide en español: '{peticion_es}'. Analiza la estancia. Devuelve JSON PRO para un cambio COMPLETO de estilo. El prompt_generacion debe estar en INGLÉS detallado."
    response = gemini_client.models.generate_content(
        model='gemini-2.5-flash', contents=[img_orig, prompt_gemini],
        config=types.GenerateContentConfig(response_mime_type="application/json", response_schema=AnalisisDecoracionPro)
    )
    datos = json.loads(response.text)
    
    # Máscara total (Blanco)
    mask_img = Image.new("L", img_orig.size, 255)
    mlsd_img = extraer_mlsd(img_orig)
    
    torch.cuda.empty_cache()
    
    img_final = pipe(
        prompt=datos['prompt_generacion'] + ", photorealistic, architectural digest, high definition",
        negative_prompt="cartoon, warped lines, messy, unrealistic lighting, deformed",
        image=img_orig, mask_image=mask_img, control_image=mlsd_img,
        num_inference_steps=25, controlnet_conditioning_scale=0.9, strength=0.95, guidance_scale=8.0
    ).images[0]
    
    return img_final, generar_html_tienda(datos)

# Pilar 2: Parcial
def motor_pilar_2_parcial(dict_imagen, peticion_es, fuerza):
    if dict_imagen is None or dict_imagen["background"] is None: return None, "Sube una imagen primero."
    
    img_orig = redimensionar(dict_imagen["background"].convert("RGB"))
    img_mascara = dict_imagen["layers"][0].split()[-1].convert("L").resize(img_orig.size, Image.Resampling.LANCZOS) if len(dict_imagen["layers"]) > 0 else Image.new("L", img_orig.size, 0)
    
    prompt_en = traducir_prompt(peticion_es)
    mlsd_img = extraer_mlsd(img_orig)
    
    torch.cuda.empty_cache()
    
    img_final = pipe(
        prompt=prompt_en + ", photorealistic, interior design, high definition, perfectly integrated",
        negative_prompt="cartoon, warped lines, messy, unrealistic lighting, deformed, mismatched style",
        image=img_orig, mask_image=img_mascara, control_image=mlsd_img,
        num_inference_steps=25, controlnet_conditioning_scale=1.0, strength=fuerza, guidance_scale=8.5
    ).images[0]
    
    # Usamos Gemini para la lista de compra parcial
    print("🛒 Generando lista de compra parcial...")
    prompt_gemini_parcial = f"Genera una lista de la compra ABUNDANTE (mín 5 productos) para lograr este diseño exacto: '{peticion_es}'."
    response = gemini_client.models.generate_content(
        model='gemini-2.5-flash', contents=[img_final, prompt_gemini_parcial],
        config=types.GenerateContentConfig(response_mime_type="application/json", response_schema=AnalisisDecoracionPro)
    )
    datos_parciales = json.loads(response.text)
    
    return img_final, generar_html_tienda(datos_parciales)

# Pilar 3: Mover (LA NOVEDAD MUNDIAL)
def motor_pilar_3_mover(dict_imagen, prompt_es):
    if dict_imagen is None or dict_imagen["background"] is None: return None, "Sube una imagen."
    
    img_orig = redimensionar(dict_imagen["background"].convert("RGB"))
    ancho, alto = img_orig.size
    
    # NUEVA UX: Múltiples Bounding Boxes (Cajas Regionales)
    regions = dict_imagen["layers"][0].convert("RGB") # Gradio V4+ devuelve cajas en layers
    mask_SAM = Image.new("L", img_orig.size, 0)
    
    SAM_detected = False
    # Gradio V4+ ImageEditor devuelve metadatos de las cajas dibujadas.
    # Esta es una simulación del SAM: convertimos la imagen pintada en máscara
    SAM_detected_img = dict_imagen["layers"][0].split()[-1]
    if SAM_detected_img.getextrema()[1] > 0:
        # Detectamos dónde ha pintado el usuario para rellenar ese hueco
        mask_SAM = SAM_detected_img.convert("L").resize(img_orig.size, Image.Resampling.LANCZOS)
        SAM_detected = True

    if not SAM_detected:
        return img_orig, "❌ Error: Debes usar la herramienta de selección regional para marcar la zona de ORIGEN y DESTINO."

    # PASO 1: Inpainting Regional (Crear el hueco de origen)
    print("🔧 Creando hueco en la zona de origen...")
    mlsd_img = extraer_mlsd(img_orig)
    
    # Motor de limpieza advanced
    img_inpaint = pipe(
        prompt="background wall and floor, photorealistic, perfectly smooth",
        negative_prompt="objects, furniture, blurry, deformed",
        image=img_orig,
        mask_image=mask_SAM,
        control_image=mlsd_img,
        num_inference_steps=20, controlnet_conditioning_scale=0.8, strength=0.90, guidance_scale=8.0
    ).images[0]

    # PASO 2: Generación Regional (Poner el mueble en el destino)
    print("🖌️ Generando mueble en la zona de destino...")
    # Creamos una máscara invertida para que la IA actúe SOLO en la zona regional SAM
    prompt_en = traducir_prompt(prompt_es)
    
    img_final = pipe(
        prompt=prompt_en + ", photorealistic, perfectly scaled, correct perspective",
        negative_prompt="cartoon, warped, messy, outdoors",
        image=img_inpaint,
        mask_image=mask_SAM, # La misma máscara sirve si la IA entiende el contexto de regional
        control_image=mlsd_img,
        num_inference_steps=25, controlnet_conditioning_scale=1.0, strength=0.85, guidance_scale=8.5
    ).images[0]
    
    # Gemini para la lista de compra del "Mover"
    prompt_gemini_mover = f"Genera una lista de compra ABUNDANTE (mín 5 productos) para este nuevo mueble: '{prompt_es}'."
    response = gemini_client.models.generate_content(
        model='gemini-2.5-flash', contents=[img_final, prompt_gemini_mover],
        config=types.GenerateContentConfig(response_mime_type="application/json", response_schema=AnalisisDecoracionPro)
    )
    datos_mover = json.loads(response.text)
    
    return img_final, generar_html_tienda(datos_mover)


# --- 6. LA INTERFAZ WEB DEFINITIVA (GRADIO 3 PESTAÑAS) ---
print("🌐 Levantando servidores web...")
with gr.Blocks(theme=gr.themes.Base(), title="DECO.AI 3.0") as app:
    gr.HTML("<center><h1>🚀 DECO.AI Studio 3.0 (Layout Edition)</h1><p>Tu Asistente Integral de Diseño de Interiores</p></center>")
    
    with gr.Tabs():
        
        # Pestaña PILAR 1: COMPLETO
        with gr.TabItem("🤖 Rediseño Total & Lista"):
            gr.Markdown("#### Pide un nuevo estilo para toda la habitación. Gemini pensará por ti.")
            with gr.Row():
                with gr.Column(scale=1):
                    in1_img = gr.Image(label="Sube tu foto")
                    in1_prompt = gr.Textbox(label="¿Qué estilo quieres?", placeholder="Ej: Cambia todo a estilo escandinavo con suelos claros")
                    btn1 = gr.Button("Analizar y Rediseñar", variant="primary")
                with gr.Column(scale=1):
                    out1_img = gr.Image(label="Resultado (Render Pro)")
            out1_html = gr.HTML(label="Lista de la Compra y Presupuesto")
            btn1.click(fn=motor_pilar_1_completo, inputs=[in1_img, in1_prompt], outputs=[out1_img, out1_html])

        # Pestaña PILAR 2: PARCIAL
        with gr.TabItem("🖌️ Cambios Selectivos (Pincel)"):
            gr.Markdown("#### Pinta los muebles que quieres cambiar (Ej: cama y mesa). Escribe lo que quieres ver allí.")
            with gr.Row():
                with gr.Column(scale=1):
                    in2_editor = gr.ImageEditor(type="pil", label="Sube foto y pinta los muebles (Pincel)")
                    in2_prompt = gr.Textbox(label="Describe los nuevos muebles", placeholder="Ej: Una cama verde terciopelo y una mesa de madera")
                    in2_fuerza = gr.Slider(minimum=0.5, maximum=1.0, value=0.85, step=0.05, label="Fuerza del cambio")
                    btn2 = gr.Button("Generar Diseño Quirúrgico", variant="primary")
                with gr.Column(scale=1):
                    out2_img = gr.Image(label="Nuevo Diseño")
            out2_html = gr.HTML(label="Lista de Productos")
            btn2.click(fn=motor_pilar_2_parcial, inputs=[in2_editor, in2_prompt, in2_fuerza], outputs=[out2_img, out2_html])

        # Pestaña PILAR 3: MOVER
        with gr.TabItem("🔧 Mover Mueble (Regional Pro)"):
            gr.Markdown("#### 🛠️ HERRAMIENTA AVANZADA: Sube la foto y usa la herramienta de selección regional (caja). Dibuja una caja donde está el mueble viejo (Origen) y otra donde quieres el nuevo (Destino).")
            with gr.Row():
                with gr.Column(scale=1):
                    # Gradio V4+ ya tiene un componente específico de ImageEditor con cajas regional
                    in3_layout = gr.ImageEditor(type="pil", tool="crop", label="⚠️ Usa el RECORTADOR Regional: Dibuja Caja A (Origen) y Caja B (Destino)")
                    in3_prompt = gr.Textbox(label="Describe el nuevo mueble (en Español)", placeholder="Ej: Una cama tapizada gris cerca de la ventana")
                    btn3 = gr.Button("Aplicar Cirugía y Mover Mueble", variant="primary")
                with gr.Column(scale=1):
                    out3_img = gr.Image(label="Mueble Movido (Render Layout)")
            out3_html = gr.HTML(label="Productos del Nuevo Mueble")
            btn3.click(fn=motor_pilar_3_mover, inputs=[in3_layout, in3_prompt], outputs=[out3_img, out3_html])

app.launch(share=True, debug=True)